# Session 2 — Your turn: Preprocessing and leakage

A bank wants to predict loan approval. The table has a column that looks useful and is not. Find it, then build the preprocessing the safe way: every number learned from the training rows only.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(3)
n = 800
loans = pd.DataFrame({"income": rng.normal(50, 15, n).clip(10, None), "age": rng.integers(21, 70, n).astype(float),
                      "debt": rng.gamma(3, 6, n), "region": rng.choice(["north", "south", "east"], n)})
score = 0.05 * loans["income"] - 0.12 * loans["debt"] + rng.normal(0, 1.2, n)
loans["approved"] = (score > np.median(score)).astype(int)
loans["decision_letter_sent"] = np.where(rng.random(n) < 0.97, loans["approved"], 1 - loans["approved"])     # only exists AFTER the decision
loans.loc[rng.choice(n, 40, replace=False), "income"] = np.nan
loans.loc[rng.choice(n, 25, replace=False), "region"] = np.nan
X, y = loans.drop(columns="approved"), loans["approved"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print(X_train.shape, X_test.shape)
from aij_check import check, close, summary

(600, 5) (200, 5)


## Task 1: Find the leak

For each column of `X_train`, fit a depth-2 decision tree on that one column alone (fill missing values with 0 and one-hot or code `region` as you like) and record its **cross-validated accuracy** in `single_column_accuracy` (a dictionary column name -> accuracy).
The column that looks too good to be true is the leak: store its name in `leaky_column`. Then fit a logistic regression on the remaining numeric columns (`income`, `age`, `debt`, missing values filled with the training median) and store its test accuracy in `honest_accuracy`.

In [2]:
from sklearn.tree import DecisionTreeClassifier
single_column_accuracy = {}
for column in X_train.columns:
    feature = X_train[[column]].copy()
    if column == "region": feature[column] = feature[column].map({"north": 0, "south": 1, "east": 2})
    feature = feature.fillna(0).to_numpy()
    single_column_accuracy[column] = cross_val_score(DecisionTreeClassifier(max_depth=2, random_state=0), feature, y_train, cv=5).mean()
leaky_column = max(single_column_accuracy, key=single_column_accuracy.get)
numeric = ["income", "age", "debt"]
median = X_train[numeric].median()
honest = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[numeric].fillna(median), y_train)
honest_accuracy = honest.score(X_test[numeric].fillna(median), y_test)
print(single_column_accuracy, leaky_column, honest_accuracy)

{'income': np.float64(0.6366666666666667), 'age': np.float64(0.5), 'debt': np.float64(0.7616666666666667), 'region': np.float64(0.47833333333333333), 'decision_letter_sent': np.float64(0.9616666666666666)} decision_letter_sent 0.805


In [3]:
check("one accuracy per column", lambda: set(single_column_accuracy) == set(X_train.columns) and all(isinstance(v, (float, np.floating)) for v in single_column_accuracy.values()))
check("the leaky column is found", lambda: leaky_column == "decision_letter_sent", "Which single column predicts the outcome almost perfectly?")
check("the leak scores above 0.9 alone", lambda: single_column_accuracy["decision_letter_sent"] > 0.9)
check("the honest model scores clearly less, but above chance", lambda: 0.6 < honest_accuracy < 0.9, "Use only income, age and debt, and fill the gaps with the TRAINING median.")
summary()

PASS  one accuracy per column
PASS  the leaky column is found
PASS  the leak scores above 0.9 alone
PASS  the honest model scores clearly less, but above chance

4 of 4 checks passed.  Well done.


**Why it works.** A column that is only known after the outcome (the decision letter) predicts it almost perfectly and would be missing at prediction time. The honest accuracy is lower, and it is the number that will hold in use.

## Task 2: A pipeline that learns only from the training rows

Build `pipe` with `make_pipeline`: a `ColumnTransformer` that handles the numeric columns (`income`, `age`, `debt`: median imputation, then standard scaling) and the `region` column (most-frequent imputation, then one-hot encoding with `handle_unknown="ignore"`), followed by a `LogisticRegression(max_iter=1000)`.
Use the default step names that `make_pipeline` gives. Drop the leaky column first. Compute 5-fold cross-validated accuracy on the training rows in `cv_scores`, then fit once and store the test accuracy in `test_accuracy`.

In [4]:
features = ["income", "age", "debt", "region"]
numeric_steps = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
region_steps = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))
pipe = make_pipeline(ColumnTransformer([("num", numeric_steps, ["income", "age", "debt"]), ("cat", region_steps, ["region"])]), LogisticRegression(max_iter=1000))
cv_scores = cross_val_score(pipe, X_train[features], y_train, cv=5)
pipe.fit(X_train[features], y_train)
test_accuracy = pipe.score(X_test[features], y_test)
print(cv_scores.round(3), test_accuracy)

[0.792 0.858 0.742 0.783 0.808] 0.8


In [5]:
check("five cross-validation scores", lambda: len(cv_scores) == 5)
check("cross-validated accuracy is believable (0.6 to 0.9)", lambda: 0.6 < np.mean(cv_scores) < 0.9)
check("the test accuracy is close to the cross-validated one", lambda: abs(test_accuracy - np.mean(cv_scores)) < 0.1)
check("the imputer learned the TRAINING median of income", lambda: close(pipe[0].named_transformers_["num"][0].statistics_[0], X_train["income"].median(), 1e-9),
      "Fit the pipeline on X_train only; the test rows must not influence any learned number.")
summary()

PASS  five cross-validation scores
PASS  cross-validated accuracy is believable (0.6 to 0.9)
PASS  the test accuracy is close to the cross-validated one
PASS  the imputer learned the TRAINING median of income

4 of 4 checks passed.  Well done.


**Why it works.** Inside a pipeline, `fit` learns the medians, the scaling and the one-hot categories from the training rows only, and `score` applies them to the test rows. Cross-validation refits the whole pipeline in every fold, so even the validation folds never leak into the numbers learned.